# 05 - ML Forecasting & High-Use Alerts

I first used a simple weekly baseline, then compared tree-based models and a small ANN.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
hourly = pd.read_csv(
    "../01_Data/processed/model_ready_hourly.csv",
    parse_dates=["date_time"]
)


In [ ]:
features = ["hour", "dayofweek", "month", "is_weekend",
            "lag_1", "lag_24", "lag_168", "rolling_24", "rolling_168"]

## Time based split

I did not shuffle the data. The last 20% is the test period.

In [ ]:
cut = int(len(hourly) * 0.8)
train = hourly.iloc[:cut].copy()
test = hourly.iloc[cut:].copy()

## Weekly baseline

In [ ]:
pred_base = test["lag_168"]
y_test = test["energy_kwh"]

In [ ]:
base_mae = mean_absolute_error(y_test, pred_base)
print("Baseline MAE:", round(base_mae, 3))

## Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=120, max_depth=12, random_state=42, n_jobs=-1
)

In [ ]:
rf.fit(train[features], train["energy_kwh"])
pred_rf = rf.predict(test[features])

In [ ]:
rf_mae = mean_absolute_error(y_test, pred_rf)
rf_rmse = mean_squared_error(y_test, pred_rf) ** 0.5
rf_r2 = r2_score(y_test, pred_rf)

In [ ]:
print("RF MAE:", round(rf_mae, 3))
print("RF RMSE:", round(rf_rmse, 3))
print("RF R2:", round(rf_r2, 3))

## Gradient Boosting benchmark

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=100, max_depth=2, learning_rate=0.04, random_state=42
)

In [ ]:
gb.fit(train[features], train["energy_kwh"])
pred_gb = gb.predict(test[features])

In [ ]:
print("GB MAE:", round(mean_absolute_error(y_test, pred_gb), 3))
print("GB R2:", round(r2_score(y_test, pred_gb), 3))

## Small ANN benchmark

In [ ]:
ann = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(32,16), max_iter=250,
                 early_stopping=True, random_state=42)
)

In [ ]:
ann.fit(train[features], train["energy_kwh"])
pred_ann = ann.predict(test[features])

In [ ]:
print("ANN MAE:", round(mean_absolute_error(y_test, pred_ann), 3))
print("ANN R2:", round(r2_score(y_test, pred_ann), 3))

## Actual vs predicted

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(test["date_time"].tail(168), y_test.tail(168), label="Actual")
plt.plot(test["date_time"].tail(168), pred_rf[-168:], label="Random Forest")
plt.legend()
plt.ylabel("kWh")
plt.title("Actual vs Forecast")
plt.tight_layout()
plt.show()

In [ ]:
plt.savefig("../04_Visuals/forecasting/10_forecast_actual_vs_pred.png", dpi=150)
plt.show()


## High-use day alert

Next I used daily energy to flag days that are above the normal high-use level.

In [ ]:
daily = pd.read_csv("../01_Data/processed/daily_energy.csv", parse_dates=["date"])
daily = daily.sort_values("date")


In [ ]:
daily["lag_1"] = daily["daily_kwh"].shift(1)
daily["lag_7"] = daily["daily_kwh"].shift(7)


In [ ]:
daily["rolling_7"] = daily["daily_kwh"].shift(1).rolling(7).mean()
daily = daily.dropna().copy()


In [ ]:
day_features = ["dayofweek", "month", "is_weekend", "lag_1", "lag_7", "rolling_7"]
cut = int(len(daily) * 0.8)
train = daily.iloc[:cut].copy()
test = daily.iloc[cut:].copy()


In [ ]:
limit = train["daily_kwh"].quantile(0.75)
train["high_use"] = (train["daily_kwh"] > limit).astype(int)
test["high_use"] = (test["daily_kwh"] > limit).astype(int)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [ ]:
dt = DecisionTreeClassifier(max_depth=2, random_state=42)
dt.fit(train[day_features], train["high_use"])
pred_dt = dt.predict(test[day_features])


In [ ]:
rf_alert = RandomForestClassifier(
    n_estimators=120, max_depth=5, class_weight="balanced",
    random_state=42, n_jobs=-1
)


In [ ]:
rf_alert.fit(train[day_features], train["high_use"])
pred_alert = rf_alert.predict(test[day_features])


In [ ]:
print("Accuracy:", round(accuracy_score(test["high_use"], pred_alert), 3))
print("Precision:", round(precision_score(test["high_use"], pred_alert), 3))
print("Recall:", round(recall_score(test["high_use"], pred_alert), 3))
print("F1:", round(f1_score(test["high_use"], pred_alert), 3))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(test["high_use"], pred_alert)
plt.title("High-Use Day Alerts")
plt.tight_layout()
plt.savefig("../04_Visuals/ml/09_confusion_matrix.png", dpi=150)
plt.show()


The alert model is a simple monitoring step. It helps find high-use days; it does not directly prove a saving in the electricity bill.